In [19]:
import pandas as pd

In [20]:
projects_data = pd.ExcelFile("243 проекты.xlsx")
financial_data = pd.ExcelFile("Фин данные ДАЙС.xlsx")

projects_data.sheet_names, financial_data.sheet_names

(['Sheet1'], ['Лист1'])

In [21]:
df_projects = projects_data.parse('Sheet1')
df_financial = financial_data.parse('Лист1')

df_projects.head(), df_financial.head()

(   Unnamed: 0.1  Unnamed: 0  gara_id                application_name  \
 0             0           0   729352  Заявка на гарантирование - 335   
 1             1           1   790750  Заявка на гарантирование - 334   
 2             2           2   845821  Заявка на гарантирование - 407   
 3             3           3   781580  Заявка на гарантирование - 379   
 4             4           4   746713  Заявка на гарантирование - 373   
 
       author_id    builder_id  contractor_id  designer_id     engine_id  \
 0  5.400012e+08  960340000376   9.603400e+11          0.0  2.210400e+11   
 1  4.044001e+10  960340000376   9.603400e+11          0.0  2.210400e+11   
 2  8.034001e+10   60340004172   6.034000e+10          0.0  1.304400e+11   
 3  5.114001e+10  981040003722   9.810400e+11          0.0  7.054001e+10   
 4  9.207400e+11   80440009616   8.044001e+10          0.0  2.208400e+11   
 
      manager_id  ... flat_height   dpg_date dpg_num  dpg_add_num  \
 0  1.904400e+11  ...         NaN

In [22]:
df_financial_pivot = df_financial.pivot_table(
    values='value', 
    index=['bin', 'date_id', 'date'], 
    columns='name', 
    aggfunc='sum'
).reset_index()

df_financial_pivot.head()

name,bin,date_id,date,I. ДДС от операционной деятельности,II. ДДС от инвестиционной деятельности,III. ДДС от финансовой деятельности,Авансы выданные,Авансы полученные,"Авансы, полученные от покупателей",Активы по отложенному подоходному налогу,...,Текущие налоговые активы,Уставный капитал,вознаграждения работникам,долгосрочные гарантийные обязательства,"инвестиции, учитываемые методом долевого участия","краткосрочные финансовые активы, оцениваемые по амортизированной стоимости",неоплаченный капитал,отложенные налоговые активы,прочая краткосрочная задолженность,прочие долгосрочные финансовые активы
0,7.400024e+08,year-1,2022-12-31,241579.0,50069.0,-586500.0,NaN,NaN,1.985715e+06,NaN,...,32016.0,2300.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,7.400024e+08,year-2,2021-12-31,494154.0,1703.0,-448128.0,NaN,NaN,2.018583e+06,NaN,...,28974.0,2300.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.034000e+10,year-1,2022-12-31,-108967693.0,-50853313.0,124732324.0,NaN,NaN,1.159823e+09,NaN,...,0.0,77500.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.034000e+10,year-2,2021-12-31,-798802365.0,-32063820.0,817208793.0,NaN,NaN,0.000000e+00,NaN,...,0.0,77500.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.034000e+10,year_current,2023-12-31,212862122.0,-328008982.0,174826571.0,NaN,NaN,0.000000e+00,NaN,...,3528425.0,77500.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
essential_columns = [
    'bin', 'date_id', 'date', 'Итого краткосрочные активы', 'Итого краткосрочных обязательств',
    'Денежные средства', 'Краткосрочная дебиторская задолженность', 'Запасы', 'Себестоимость',
    'Доход от реализации', 'Итого активы', 'Итого капитал', 'Итого пассивы',
    'Краткосрочная кредиторская задолженность', 'Операционная прибыль',
    'Долгосрочные финансовые обязательства', 'Краткосрочные финансовые обязательства',
    'Итоговая прибыль (убыток)', 'Расходы на финансирование'
]

df_filtered = df_financial_pivot[essential_columns].copy()

df_filtered['Коэффициент текущей ликвидности'] = (
    df_filtered['Итого краткосрочные активы'] / df_filtered['Итого краткосрочных обязательств']
)

df_filtered['Коэффициент быстрой ликвидности'] = (
    (df_filtered['Краткосрочная дебиторская задолженность'] + df_filtered['Денежные средства']) / 
    df_filtered['Итого краткосрочных обязательств']
)

df_filtered['Коэффициент абсолютной ликвидности'] = (
    df_filtered['Денежные средства'] / df_filtered['Итого краткосрочных обязательств']
)

df_filtered['Коэффициент оборачиваемости запасов'] = (
    df_filtered['Себестоимость'] / df_filtered['Запасы']
)

df_filtered['Коэффициент оборачиваемости активов'] = (
    df_filtered['Доход от реализации'] / df_filtered['Итого активы']
)

df_filtered['Коэффициент оборачиваемости кредиторской задолженности'] = (
    df_filtered['Себестоимость'] / df_filtered['Краткосрочная кредиторская задолженность']
)

df_filtered['ROA'] = (
    df_filtered['Итоговая прибыль (убыток)'] / df_filtered['Итого активы']
)

df_filtered['ROE'] = (
    df_filtered['Итоговая прибыль (убыток)'] / df_filtered['Итого капитал']
)

df_filtered['Коэффициент чистой прибыли'] = (
    df_filtered['Итоговая прибыль (убыток)'] / df_filtered['Доход от реализации']
)

df_filtered['Коэффициент финансового левериджа'] = (
    df_filtered['Итого пассивы'] / df_filtered['Итого капитал']
)

df_filtered['Коэффициент финансовой независимости'] = (
    df_filtered['Итого капитал'] / df_filtered['Итого активы']
)

df_filtered['Коэффициент долговой нагрузки'] = (
    df_filtered['Итого пассивы'] / df_filtered['Итого активы']
)

df_filtered['Коэффициент долг на капитал'] = (
    (df_filtered['Краткосрочные финансовые обязательства'] + 
     df_filtered['Долгосрочные финансовые обязательства']) / 
    df_filtered['Итого капитал']
)

df_filtered['Коэффициент покрытия процентов'] = (
    df_filtered['Операционная прибыль'] / df_filtered['Расходы на финансирование']
)

df_filtered['Покрытие долга'] = (
    df_filtered['Операционная прибыль'] / 
    (df_filtered['Краткосрочные финансовые обязательства'] + df_filtered['Долгосрочные финансовые обязательства'])
)

df_filtered.head()

name,bin,date_id,date,Итого краткосрочные активы,Итого краткосрочных обязательств,Денежные средства,Краткосрочная дебиторская задолженность,Запасы,Себестоимость,Доход от реализации,...,Коэффициент оборачиваемости кредиторской задолженности,ROA,ROE,Коэффициент чистой прибыли,Коэффициент финансового левериджа,Коэффициент финансовой независимости,Коэффициент долговой нагрузки,Коэффициент долг на капитал,Коэффициент покрытия процентов,Покрытие долга
0,7.400024e+08,year-1,2022-12-31,7.948640e+05,4.581000e+05,39799.0,5.063870e+05,1.959640e+05,5.123539e+06,6.150560e+06,...,11.866307,0.058879,0.158792,0.029054,2.696941,0.370790,1.0,0.300390,3.154109,0.690626
1,7.400024e+08,year-2,2021-12-31,1.026339e+06,3.506130e+05,408671.0,3.597250e+05,2.095320e+05,6.226864e+06,7.541316e+06,...,18.995287,0.053273,0.195293,0.024515,3.665914,0.272783,1.0,0.951870,2.174446,0.391017
2,1.034000e+10,year-1,2022-12-31,5.506720e+09,2.864408e+09,3325026.0,2.763283e+09,2.544579e+09,8.559005e+09,8.777198e+09,...,3.675631,0.003545,0.017766,0.002665,5.012058,0.199519,1.0,0.379265,1.427051,0.154531
3,1.034000e+10,year-2,2021-12-31,3.976049e+09,2.060613e+09,38413708.0,1.042274e+09,1.897019e+09,3.989398e+09,4.253717e+09,...,2.488090,0.020863,0.080508,0.024478,3.858868,0.259143,1.0,1.471959,29503.539123,0.052291
4,1.034000e+10,year_current,2023-12-31,6.177684e+09,3.909230e+09,63004737.0,2.594795e+09,3.324581e+09,5.439741e+09,6.253281e+09,...,1.401120,0.060436,0.268761,0.077376,4.447005,0.224870,1.0,1.275603,3015.719330,0.210763


In [24]:
df_merged = pd.merge(
    df_projects,
    df_filtered,
    how='left',
    left_on=['builder_id', 'start_date'],
    right_on=['bin', 'date']
)

df_merged

,Unnamed: 0.1,Unnamed: 0,gara_id,application_name,author_id,builder_id,contractor_id,designer_id,engine_id,manager_id,...,Коэффициент оборачиваемости кредиторской задолженности,ROA,ROE,Коэффициент чистой прибыли,Коэффициент финансового левериджа,Коэффициент финансовой независимости,Коэффициент долговой нагрузки,Коэффициент долг на капитал,Коэффициент покрытия процентов,Покрытие долга
0,0,0,729352,Заявка на гарантирование - 335,5.400012e+08,960340000376,9.603400e+11,0.000000e+00,2.210400e+11,1.904400e+11,...,1.825000,0.009441,0.016491,0.049848,1.746798,0.572476,1.0,0.291062,0.179593,0.048677
1,1,1,790750,Заявка на гарантирование - 334,4.044001e+10,960340000376,9.603400e+11,0.000000e+00,2.210400e+11,1.904400e+11,...,1.825000,0.009441,0.016491,0.049848,1.746798,0.572476,1.0,0.291062,0.179593,0.048677
2,2,2,845821,Заявка на гарантирование - 407,8.034001e+10,60340004172,6.034000e+10,0.000000e+00,1.304400e+11,2.109400e+11,...,3.944488,0.129294,0.656131,0.068909,5.074722,0.197055,1.0,0.096349,14.272238,7.247071
3,3,3,781580,Заявка на гарантирование - 379,5.114001e+10,981040003722,9.810400e+11,0.000000e+00,7.054001e+10,2.109400e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,4,746713,Заявка на гарантирование - 373,9.207400e+11,80440009616,8.044001e+10,0.000000e+00,2.208400e+11,2.202400e+11,...,1.907142,0.466784,0.625282,4.384301,1.339554,0.746517,1.0,0.305722,inf,0.151482
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,237,239,127,NaN,NaN,30740000258,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
260,238,240,2,NaN,3.114000e+10,111140016064,2.064000e+10,NaN,1.402400e+11,2.106400e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
261,239,241,131,NaN,7.074000e+10,901040000029,9.010400e+11,7.074000e+10,6.064000e+10,1.808400e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
262,240,242,1318319,Заявка на гарантирование - 493,1.704400e+11,131240018048,1.312400e+11,0.000000e+00,2.003400e+11,1.709400e+11,...,6.713175,0.093017,0.388849,0.071243,4.180396,0.239212,1.0,0.268664,26.721384,1.503592


In [83]:
len(df_merged['Покрытие долга'].dropna())


117

In [25]:
df_merged.to_excel('243 проекта с кофф.xlsx')

In [48]:
target = pd.read_excel('Full_Data_With_Life_Bin_Backlog_Target.xlsx', sheet_name='Лист1')
persent1 = target['target 1']

In [46]:
data = pd.read_excel('Full_Data_With_Life_Bin_Backlog_Target.xlsx', sheet_name='Sheet1')
data = data[data['ID_Project'].isin(persent1)]

In [28]:
cost = data['Первоначальная стоимость проекта, тенге']

In [32]:
import numpy as np

df_merged['target'] = np.nan
df_merged['target'] = np.where(df_merged['cost_plan'].isin(cost), 0.3, df_merged['target'])

In [33]:
persent1 = target['target 1.1']

In [34]:
data = pd.read_excel('Full_Data_With_Life_Bin_Backlog_Target.xlsx', sheet_name='Sheet1')
data = data[data['ID_Project'].isin(persent1)]

In [35]:
cost = data['Первоначальная стоимость проекта, тенге']

In [36]:
df_merged['target'] = np.where(df_merged['cost_plan'].isin(cost), 0.5, df_merged['target'])

In [38]:
df_merged.to_excel('Final_data.xlsx')

In [53]:
f = target['cost ']

In [56]:
data = data[~data['Первоначальная стоимость проекта, тенге'].isin(f)]


In [57]:
data

,ID_Project,Период,rep_date,life_b,life_e,term,period_%,Название проекта (база ДТЭМ),Название проекта (база ДРЮЛ),Название проекта (1C),...,площадь ДДУ кладовые,Стоимость ДДУ кладовые,оплачено ДДУ кладовые,Отставание ГПР в % от общей прод. Строительства,Месяцев от начала строительства до заключения гарантии,"Нормативный срок строительства, месяцев",БИН Застройщика,Средний Балл Застройщика,Life_Bin,Backlog
1968,178,202312,2023-12-30,2023-07-24,2024-07-19,12.033333,0.440443,ЖК Салем***,МЖК Салем,NaN,...,NaN,NaN,NaN,0.100310,2.866667,10.166667,1.407400e+11,77.0,50%,10%+
2015,178,202401,2024-01-30,2023-07-24,2024-07-19,12.033333,0.526316,ЖК Салем,МЖК Салем,NaN,...,NaN,NaN,NaN,0.175662,2.866667,10.166667,1.407400e+11,77.0,60%,10%+
2063,178,202402,2024-02-28,2023-07-24,2024-07-19,12.033333,0.606648,ЖК Салем,МЖК Салем,NaN,...,NaN,NaN,NaN,0.253861,2.866667,10.166667,1.407400e+11,77.0,70%,30%+
2109,178,202403,2024-03-30,2023-07-24,2024-07-19,12.033333,0.692521,ЖК Салем,МЖК Салем,NaN,...,NaN,NaN,NaN,0.338215,2.866667,10.166667,1.407400e+11,77.0,70%,30%+
2161,178,202404,2024-04-30,2023-07-24,2024-07-19,12.033333,0.778393,ЖК Салем,МЖК Салем,МЖК в г.Кокшетау,...,NaN,NaN,NaN,0.426292,2.866667,10.166667,1.407400e+11,77.0,80%,30%+
2219,178,202405,2024-05-30,2023-07-24,2024-07-19,12.033333,0.861496,МЖК Салем в г.Кокшетау,МЖК Салем,"ЖК ""Salem""",...,NaN,NaN,NaN,0.515235,2.866667,9.033333,1.407400e+11,77.0,90%,50%+
2274,178,202406,2024-07-01,2023-07-24,2024-07-19,12.033333,0.950139,"ЖК ""Salem""",МЖК Салем,"ЖК ""Salem""",...,NaN,NaN,NaN,0.387812,3.033333,10.166667,1.407400e+11,77.0,100%+,50%+
2325,178,202407,2024-08-01,2023-07-24,2024-07-19,12.033333,1.036011,"ЖК ""Salem""",МЖК Салем,"ЖК ""Salem""",...,NaN,NaN,NaN,0.651940,2.866667,10.166667,1.407400e+11,77.0,100%+,50%+


In [58]:
df_merged = pd.read_excel('243 проекта с кофф.xlsx')

In [59]:
df_merged

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,gara_id,application_name,author_id,builder_id,contractor_id,designer_id,engine_id,...,Коэффициент оборачиваемости кредиторской задолженности,ROA,ROE,Коэффициент чистой прибыли,Коэффициент финансового левериджа,Коэффициент финансовой независимости,Коэффициент долговой нагрузки,Коэффициент долг на капитал,Коэффициент покрытия процентов,Покрытие долга
0,0,0,0,729352,Заявка на гарантирование - 335,5.400012e+08,960340000376,9.603400e+11,0.000000e+00,2.210400e+11,...,1.825000,0.009441,0.016491,0.049848,1.746798,0.572476,1.0,0.291062,0.179593,0.048677
1,1,1,1,790750,Заявка на гарантирование - 334,4.044001e+10,960340000376,9.603400e+11,0.000000e+00,2.210400e+11,...,1.825000,0.009441,0.016491,0.049848,1.746798,0.572476,1.0,0.291062,0.179593,0.048677
2,2,2,2,845821,Заявка на гарантирование - 407,8.034001e+10,60340004172,6.034000e+10,0.000000e+00,1.304400e+11,...,3.944488,0.129294,0.656131,0.068909,5.074722,0.197055,1.0,0.096349,14.272238,7.247071
3,3,3,3,781580,Заявка на гарантирование - 379,5.114001e+10,981040003722,9.810400e+11,0.000000e+00,7.054001e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,4,4,746713,Заявка на гарантирование - 373,9.207400e+11,80440009616,8.044001e+10,0.000000e+00,2.208400e+11,...,1.907142,0.466784,0.625282,4.384301,1.339554,0.746517,1.0,0.305722,inf,0.151482
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,259,237,239,127,NaN,NaN,30740000258,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
260,260,238,240,2,NaN,3.114000e+10,111140016064,2.064000e+10,NaN,1.402400e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
261,261,239,241,131,NaN,7.074000e+10,901040000029,9.010400e+11,7.074000e+10,6.064000e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
262,262,240,242,1318319,Заявка на гарантирование - 493,1.704400e+11,131240018048,1.312400e+11,0.000000e+00,2.003400e+11,...,6.713175,0.093017,0.388849,0.071243,4.180396,0.239212,1.0,0.268664,26.721384,1.503592


In [65]:
gd = df_merged['gara_id'].unique().tolist()


In [73]:
len(gd)

241

In [69]:
financial_data = pd.ExcelFile("Фин данные ДАЙС.xlsx")
df_financial = financial_data.parse("Лист1")

In [70]:
gd1 = df_financial['gara_id'].unique().tolist()


In [72]:
len(gd1)

151

In [79]:
lost_projects = [project for project in gd if project not in gd1]


In [81]:
lost_projects

[157,
 85,
 114,
 15,
 57,
 14,
 51,
 135,
 80,
 25,
 58,
 17,
 113,
 86,
 4,
 140,
 123,
 129,
 99,
 103,
 6,
 109,
 71,
 161,
 9,
 87,
 52,
 20,
 49,
 156,
 136,
 179,
 23,
 18,
 153,
 45,
 88,
 75,
 186,
 46,
 50,
 133,
 197,
 39,
 78,
 53,
 81,
 59,
 3,
 146,
 148,
 110,
 40,
 76,
 174,
 73,
 149,
 128,
 195,
 19,
 38,
 83,
 124,
 163,
 35,
 190,
 94,
 145,
 175,
 70,
 112,
 62,
 74,
 56,
 26,
 43,
 69,
 77,
 96,
 10,
 21,
 1,
 151,
 202,
 66,
 55,
 54,
 106,
 162,
 65,
 12,
 16,
 155,
 125,
 67,
 11,
 22,
 115,
 37,
 170,
 117,
 189,
 164,
 93,
 90,
 159,
 147,
 33,
 60,
 64,
 127]